# SofaScore Match ID Extractor

This notebook loads a SofaScore round through undetected Chrome, extracts the match and team IDs, and saves the results to `match_ids_{match_day}.json`.


In [1]:
# Resolve the project root and import authoritative data locations.
import sys
from pathlib import Path


# Handle project root for reuse in the workflow.
def _locate_project_root() -> Path:
    starts = []
    vscode_notebook = globals().get("__vsc_ipynb_file__")
    if isinstance(vscode_notebook, str) and vscode_notebook.strip():
        starts.append(Path(vscode_notebook).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())

    checked = set()
    # Process each available item while preserving the current workflow state.
    for start in starts:
        # Process each available item while preserving the current workflow state.
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.add(candidate)
            if (candidate / "project_paths.py").is_file():
                return candidate
    raise FileNotFoundError(
        "Could not locate project_paths.py. Start Jupyter from the Kickbase "
        "project root or open this notebook from within that project."
    )


# Set workflow configuration value: _PROJECT_ROOT.
_PROJECT_ROOT = _locate_project_root()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from project_paths import (
    SOFASCORE_MATCH_IDS_DIR,
    ensure_directory,
)


In [2]:
# Install the dependency required for the following notebook steps.
%pip install undetected-chromedriver


Note: you may need to restart the kernel to use updated packages.


In [3]:
# Import the libraries required by this notebook step.
import json
from pathlib import Path
from typing import Any

import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait


# Set workflow configuration value: MATCH_DAY.
MATCH_DAY = 1
# Set workflow configuration value: ENDPOINT_URL_TEMPLATE.
ENDPOINT_URL_TEMPLATE = (
    "https://www.sofascore.com/api/v1/unique-tournament/35/"
    "season/97464/events/round/{match_day}"
)
# Set workflow configuration value: OUTPUT_PATH_TEMPLATE.
OUTPUT_PATH_TEMPLATE = "match_ids_{match_day}.json"
# Set workflow configuration value: CHROME_MAJOR_VERSION.
CHROME_MAJOR_VERSION = None  # Let undetected-chromedriver auto-detect Chrome.
# Set workflow configuration value: WAIT_TIMEOUT_SECONDS.
WAIT_TIMEOUT_SECONDS = 30


# Define Extraction Error to keep related behaviour explicit.
class ExtractionError(RuntimeError):
    """Raised when match data cannot be retrieved or validated."""


In [4]:
# Retrieve response text for reuse in the workflow.
def fetch_response_text(url: str) -> str:
    """Load a URL through undetected Chrome and return its body text."""
    options = uc.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")

    driver = None
    # Handle expected failures with a clear, actionable message.
    try:
        # Handle expected failures with a clear, actionable message.
        try:
            driver = uc.Chrome(
                options=options,
                            )
        except Exception as exc:
            raise ExtractionError(
                f"Could not start Chrome {CHROME_MAJOR_VERSION}: {exc}"
            ) from exc

        # Handle expected failures with a clear, actionable message.
        try:
            driver.set_page_load_timeout(WAIT_TIMEOUT_SECONDS)
            driver.get(url)
            body = WebDriverWait(driver, WAIT_TIMEOUT_SECONDS).until(
                lambda current_driver: (
                    element
                    if (element := current_driver.find_element(By.TAG_NAME, "body"))
                    and element.text.strip()
                    else False
                )
            )
        except TimeoutException as exc:
            raise ExtractionError(
                f"Timed out after {WAIT_TIMEOUT_SECONDS} seconds waiting for "
                "the SofaScore response."
            ) from exc
        except Exception as exc:
            raise ExtractionError(
                f"Could not load the SofaScore endpoint: {exc}"
            ) from exc

        response_text = body.text.strip()
        # Validate the input before continuing with later processing.
        if not response_text:
            raise ExtractionError("SofaScore returned an empty response body.")
        return response_text
    finally:
        if driver is not None:
            # Handle expected failures with a clear, actionable message.
            try:
                driver.quit()
            except Exception:
                pass


In [5]:
# Parse and validate matches for reuse in the workflow.
def parse_matches(response_text: str) -> list[dict[str, Any]]:
    """Parse and validate matchups from a SofaScore JSON response."""
    # Handle expected failures with a clear, actionable message.
    try:
        payload = json.loads(response_text)
    except json.JSONDecodeError as exc:
        raise ExtractionError(
            "SofaScore did not return valid JSON "
            f"(line {exc.lineno}, column {exc.colno})."
        ) from exc

    # Validate the input before continuing with later processing.
    if not isinstance(payload, dict):
        raise ExtractionError("The SofaScore response must be a JSON object.")
    # Validate the input before continuing with later processing.
    if "events" not in payload:
        raise ExtractionError(
            "The SofaScore response does not contain an 'events' field."
        )

    events = payload["events"]
    # Validate the input before continuing with later processing.
    if not isinstance(events, list):
        raise ExtractionError("The SofaScore 'events' field must be a list.")

    matches: list[dict[str, Any]] = []
    # Process each available item while preserving the current workflow state.
    for index, event in enumerate(events, start=1):
        # Validate the input before continuing with later processing.
        if not isinstance(event, dict):
            raise ExtractionError(f"Event {index} is not a JSON object.")

        home_team = event.get("homeTeam")
        away_team = event.get("awayTeam")
        match_id = event.get("id")

        # Validate the input before continuing with later processing.
        if not isinstance(home_team, dict) or not isinstance(
            home_team.get("name"), str
        ):
            raise ExtractionError(f"Event {index} has no valid homeTeam.name.")
        # Validate the input before continuing with later processing.
        if not isinstance(away_team, dict) or not isinstance(
            away_team.get("name"), str
        ):
            raise ExtractionError(f"Event {index} has no valid awayTeam.name.")
        # Validate the input before continuing with later processing.
        if not isinstance(match_id, int) or isinstance(match_id, bool):
            raise ExtractionError(f"Event {index} has no valid integer match ID.")

        home_team_id = home_team.get("id")
        away_team_id = away_team.get("id")
        # Validate the input before continuing with later processing.
        if not isinstance(home_team_id, int) or isinstance(home_team_id, bool):
            raise ExtractionError(f"Event {index} has no valid homeTeam.id.")
        # Validate the input before continuing with later processing.
        if not isinstance(away_team_id, int) or isinstance(away_team_id, bool):
            raise ExtractionError(f"Event {index} has no valid awayTeam.id.")

        home_name = home_team["name"].strip()
        away_name = away_team["name"].strip()
        # Validate the input before continuing with later processing.
        if not home_name:
            raise ExtractionError(f"Event {index} has an empty home team name.")
        # Validate the input before continuing with later processing.
        if not away_name:
            raise ExtractionError(f"Event {index} has an empty away team name.")

        matches.append(
            {
                "home_team": home_name,
                "home_team_id": home_team_id,
                "away_team": away_name,
                "away_team_id": away_team_id,
                "match_id": match_id,
            }
        )

    return matches


# Save matches for reuse in the workflow.
def save_matches(matches: list[dict[str, Any]], output_path: Path) -> None:
    """Write extracted matches to a UTF-8 JSON file."""
    # Handle expected failures with a clear, actionable message.
    try:
        output_path.write_text(
            json.dumps(matches, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )
    except OSError as exc:
        raise ExtractionError(f"Could not write {output_path}: {exc}") from exc


# Extract match IDs for reuse in the workflow.
def extract_match_ids(match_day: int) -> list[dict[str, Any]]:
    """Fetch, validate, display, and save matches for one matchday."""
    # Validate the input before continuing with later processing.
    if not isinstance(match_day, int) or isinstance(match_day, bool):
        raise ValueError("match_day must be an integer.")
    # Validate the input before continuing with later processing.
    if match_day < 1:
        raise ValueError("match_day must be at least 1.")

    endpoint_url = ENDPOINT_URL_TEMPLATE.format(match_day=match_day)
    output_path = ensure_directory(SOFASCORE_MATCH_IDS_DIR) / OUTPUT_PATH_TEMPLATE.format(match_day=match_day)
    response_text = fetch_response_text(endpoint_url)
    matches = parse_matches(response_text)
    save_matches(matches, output_path)

    # Process each available item while preserving the current workflow state.
    for match in matches:
        print(
            f"{match['home_team']} ({match['home_team_id']}) vs "
            f"{match['away_team']} ({match['away_team_id']}): "
            f"match ID {match['match_id']}"
        )

    if not matches:
        print("No matches found.")
    print(f"Saved {len(matches)} match(es) to {output_path}.")
    return matches


In [6]:
# Run this self-contained workflow step using the prepared inputs.
md = int(input("Which Bundesliga Matchday do you want to extract data from?"))


matches = extract_match_ids(match_day=md)
matches


Which Bundesliga Matchday do you want to extract data from? 2


VfB Stuttgart (2677) vs 1. FC Köln (2671): match ID 16434036
Bayer 04 Leverkusen (2681) vs 1. FC Union Berlin (2547): match ID 16434042
Borussia M'gladbach (2527) vs SV 07 Elversberg (2598): match ID 16434028
SC Paderborn 07 (2561) vs SC Freiburg (2538): match ID 16434024
SV Werder Bremen (2534) vs RB Leipzig (36360): match ID 16434101
TSG Hoffenheim (2569) vs Borussia Dortmund (2673): match ID 16434019
FC Schalke 04 (2530) vs FC Bayern München (2672): match ID 16434032
Hamburger SV (2676) vs 1. FSV Mainz 05 (2556): match ID 16434030
Eintracht Frankfurt (2674) vs FC Augsburg (2600): match ID 16434021
Saved 9 match(es) to C:\kickbase project\outputs\sofascore\match_ids\match_ids_2.json.


[{'home_team': 'VfB Stuttgart',
  'home_team_id': 2677,
  'away_team': '1. FC Köln',
  'away_team_id': 2671,
  'match_id': 16434036},
 {'home_team': 'Bayer 04 Leverkusen',
  'home_team_id': 2681,
  'away_team': '1. FC Union Berlin',
  'away_team_id': 2547,
  'match_id': 16434042},
 {'home_team': "Borussia M'gladbach",
  'home_team_id': 2527,
  'away_team': 'SV 07 Elversberg',
  'away_team_id': 2598,
  'match_id': 16434028},
 {'home_team': 'SC Paderborn 07',
  'home_team_id': 2561,
  'away_team': 'SC Freiburg',
  'away_team_id': 2538,
  'match_id': 16434024},
 {'home_team': 'SV Werder Bremen',
  'home_team_id': 2534,
  'away_team': 'RB Leipzig',
  'away_team_id': 36360,
  'match_id': 16434101},
 {'home_team': 'TSG Hoffenheim',
  'home_team_id': 2569,
  'away_team': 'Borussia Dortmund',
  'away_team_id': 2673,
  'match_id': 16434019},
 {'home_team': 'FC Schalke 04',
  'home_team_id': 2530,
  'away_team': 'FC Bayern München',
  'away_team_id': 2672,
  'match_id': 16434032},
 {'home_team':